### 1. Data Loading & Content-Based Filtering Setup
In this section, we load the processed dataset, combine relevant text features, and build the TF-IDF vectorizer along with the Cosine Similarity matrix.

In [17]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv(r'..\datasets\processed\cleaned_medicine_prepared.csv')

df['uses'] = df['uses'].fillna('')
df['composition'] = df['composition'].fillna('')
df['manufacturer'] = df['manufacturer'].fillna('')

df['combined_features'] = df['medicine_name'] + " " + df['uses'] + " " + df['composition']

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Model trained successfully")

Model trained successfully


### 2. Content-Based Recommendation Function
Defining a function to fetch top medicine recommendations based purely on text similarity (uses and composition).

In [18]:
def recommend_medicines(user_input, num_recommendations=5):
    user_input = user_input.lower()
    
    matched_indices = df[df['combined_features'].str.lower().str.contains(user_input)].index.tolist()
    
    if not matched_indices:
        return "Sorry, no medicine related to this condition or disease was found in the dataset."
    
    idx = matched_indices[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:num_recommendations+1]
    medicine_indices = [i[0] for i in sim_scores]
    
    results = df.iloc[medicine_indices][['medicine_name', 'uses', 'composition', 'manufacturer']]
    return results

print("--- Recommended Medicines ---")
print(recommend_medicines('Bacterial infection', 5))

--- Recommended Medicines ---
                  medicine_name                               uses  \
63    Augmentin 1000 Duo Tablet  Treatment of Bacterial infections   
103        Augmentin 375 Tablet  Treatment of Bacterial infections   
167           Advent 625 Tablet  Treatment of Bacterial infections   
5107         Moxclav 625 Tablet  Treatment of Bacterial infections   
1192       Bactoclav 625 Tablet  Treatment of Bacterial infections   

                                          composition  \
63    Amoxycillin  (875mg) +  Clavulanic Acid (125mg)   
103   Amoxycillin  (250mg) +  Clavulanic Acid (125mg)   
167   Amoxycillin  (500mg) +  Clavulanic Acid (125mg)   
5107  Amoxycillin  (500mg) +  Clavulanic Acid (125mg)   
1192  Amoxycillin  (500mg) +  Clavulanic Acid (125mg)   

                              manufacturer  
63    Glaxo SmithKline Pharmaceuticals Ltd  
103   Glaxo SmithKline Pharmaceuticals Ltd  
167                              Cipla Ltd  
5107     Sun Pharmaceutica

### 3. Collaborative Filtering (SVD Model) Setup
Training an SVD model using the Surprise library to predict user-item ratings based on historical review patterns.

In [19]:
from surprise import Reader, Dataset, SVD
from surprise.model_selection import cross_validate
import pandas as pd

df = pd.read_csv('../datasets/processed/cleaned_medicine_prepared.csv')

if 'user_id' not in df.columns:
    
    df['item_id'] = df.index
    df['user_id'] = 1  
reader = Reader(rating_scale=(1, 5))

df['rating'] = df['excellent_review'].apply(lambda x: 5 if x > 50 else (3 if x > 20 else 1))

data = Dataset.load_from_df(df[['user_id', 'item_id', 'rating']], reader)

algo = SVD()
print("--- Training & Evaluating SVD Collaborative Filtering Model ---")
cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=3, verbose=True)

print("Collaborative Filtering SVD setup run successfully!")

--- Training & Evaluating SVD Collaborative Filtering Model ---
Evaluating RMSE, MAE of algorithm SVD on 3 split(s).

                  Fold 1  Fold 2  Fold 3  Mean    Std     
RMSE (testset)    1.3681  1.3496  1.3632  1.3603  0.0078  
MAE (testset)     1.0071  0.9811  0.9660  0.9847  0.0170  
Fit time          0.18    0.20    0.22    0.20    0.02    
Test time         0.03    0.33    0.03    0.13    0.14    
Collaborative Filtering SVD setup run successfully!


### 4. Basic Hybrid Recommendation System
Combining Content-Based similarity scores and Collaborative Filtering predictions using weighted averaging.

In [20]:
import numpy as np
import pandas as pd

def hybrid_recommendation(medicine_idx, user_id, num_recommendations=5):
    
    content_scores = list(enumerate(cosine_sim[medicine_idx]))
    
    collab_scores = []
    for idx in range(len(df)):
        pred_rating = algo.predict(uid=user_id, iid=idx).est
        collab_scores.append((idx, pred_rating / 5.0)) # Normalize karke 0 se 1 ke beech laana
    
    weight_content = 0.6
    weight_collab = 0.4
    
    hybrid_scores = []
    for i in range(len(df)):
        c_score = content_scores[i][1]
        col_score = collab_scores[i][1]
        
        final_score = (weight_content * c_score) + (weight_collab * col_score)
        hybrid_scores.append((i, final_score))
    
    hybrid_scores = sorted(hybrid_scores, key=lambda x: x[1], reverse=True)
    
    hybrid_scores = [item for item in hybrid_scores if item[0] != medicine_idx]
    top_indices = [item[0] for item in hybrid_scores[:num_recommendations]]
    
    results = df.iloc[top_indices][['medicine_name', 'uses', 'composition', 'manufacturer']]
    return results

print("--- Hybrid Recommended Medicines ---")
print(hybrid_recommendation(medicine_idx=0, user_id=1, num_recommendations=5))

--- Hybrid Recommended Medicines ---
                          medicine_name  \
3274               Gemtaz 1gm Injection   
3073             Geftistar 250mg Tablet   
2329            Eleftha 150mg Injection   
3863           Irinotel 100mg Injection   
6426  Nindanib 100 Soft Gelatin Capsule   

                                                   uses          composition  \
3274  Non-small cell lung cancer Breast cancer Pancr...    Gemcitabine (1gm)   
3073                         Non-small cell lung cancer    Gefitinib (250mg)   
2329                       Breast cancer Stomach cancer  Trastuzumab (150mg)   
3863                         Cancer of colon and rectum   Irinotecan (100mg)   
6426  Treatment of Idiopathic pulmonary fibrosisTrea...   Nintedanib (100mg)   

                           manufacturer  
3274  Sun Pharmaceutical Industries Ltd  
3073                          Lupin Ltd  
2329          Intas Pharmaceuticals Ltd  
3863       Fresenius Kabi India Pvt Ltd  
6426       Gl

### 5. Sentiment Analysis & Review Scoring
Processing user review columns (excellent, average, poor) to calculate a normalized sentiment/quality score for each medicine.

In [21]:
import pandas as pd
import numpy as np

df['excellent_review'] = pd.to_numeric(df['excellent_review'], errors='coerce').fillna(0)
df['average_review'] = pd.to_numeric(df['average_review'], errors='coerce').fillna(0)
df['poor_review'] = pd.to_numeric(df['poor_review'], errors='coerce').fillna(0)

total_reviews = df['excellent_review'] + df['average_review'] + df['poor_review']

total_reviews = total_reviews.replace(0, 1)

df['sentiment_score'] = (
    (df['excellent_review'] * 1.0) + 
    (df['average_review'] * 0.5) + 
    (df['poor_review'] * 0.0)
) / total_reviews

print("--- Sentiment Analysis & Review Scores Successfully Calculated ---")
print(df[['medicine_name', 'excellent_review', 'poor_review', 'sentiment_score']].head())

--- Sentiment Analysis & Review Scores Successfully Calculated ---
              medicine_name  excellent_review  poor_review  sentiment_score
0   Avastin 400mg Injection                22           22            0.500
1  Augmentin 625 Duo Tablet                47           18            0.645
2       Azithral 500 Tablet                39           21            0.590
3          Ascoril LS Syrup                24           35            0.445
4         Aciloc 150 Tablet                34           29            0.525


### 6. Advanced Hybrid Recommendation Engine
Integrating Content-Based filtering, SVD Collaborative filtering, and Sentiment Analysis scores into a unified scoring formula (40% Content, 30% Collaborative, 30% Sentiment).

In [22]:
def advanced_hybrid_recommendation(medicine_idx, user_id, num_recommendations=5):
    content_scores = list(enumerate(cosine_sim[medicine_idx]))
    
    collab_scores = []
    for idx in range(len(df)):
        pred_rating = algo.predict(uid=user_id, iid=idx).est
        collab_scores.append((idx, pred_rating / 5.0))
        
    w_content = 0.4
    w_collab = 0.3
    w_sentiment = 0.3
    
    hybrid_scores = []
    for i in range(len(df)):
        c_score = content_scores[i][1]
        col_score = collab_scores[i][1]
        sent_score = df.loc[i, 'sentiment_score']
        
        final_score = (w_content * c_score) + (w_collab * col_score) + (w_sentiment * sent_score)
        hybrid_scores.append((i, final_score))
        
    hybrid_scores = sorted(hybrid_scores, key=lambda x: x[1], reverse=True)
    hybrid_scores = [item for item in hybrid_scores if item[0] != medicine_idx]
    top_indices = [item[0] for item in hybrid_scores[:num_recommendations]]
    
    results = df.iloc[top_indices][['medicine_name', 'uses', 'composition', 'sentiment_score', 'manufacturer']]
    return results

print("\n--- Advanced Hybrid Recommendations (Including Sentiment Analysis) ---")
print(advanced_hybrid_recommendation(medicine_idx=0, user_id=1, num_recommendations=5))


--- Advanced Hybrid Recommendations (Including Sentiment Analysis) ---
                 medicine_name  \
3274      Gemtaz 1gm Injection   
2329   Eleftha 150mg Injection   
3863  Irinotel 100mg Injection   
3073    Geftistar 250mg Tablet   
682             Anabrez Tablet   

                                                   uses          composition  \
3274  Non-small cell lung cancer Breast cancer Pancr...    Gemcitabine (1gm)   
2329                       Breast cancer Stomach cancer  Trastuzumab (150mg)   
3863                         Cancer of colon and rectum   Irinotecan (100mg)   
3073                         Non-small cell lung cancer    Gefitinib (250mg)   
682                                       Breast cancer    Anastrozole (1mg)   

      sentiment_score                       manufacturer  
3274            1.000  Sun Pharmaceutical Industries Ltd  
2329            1.000          Intas Pharmaceuticals Ltd  
3863            1.000       Fresenius Kabi India Pvt Ltd  
3073  

### 7. Model Persistence & Saving
Saving all trained models (Vectorizer, Similarity Matrix, SVD Algorithm, and Processed DataFrame) into the `../models/` directory using Joblib for deployment.

In [ ]:
import joblib
import os

# 1. Models aur processed files ke liye folder ensure karna
os.makedirs('../models', exist_ok=True)

# 2. Saare main components ko save karna
joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')
joblib.dump(cosine_sim, '../models/cosine_similarity.pkl')
joblib.dump(algo, '../models/svd_collaborative_model.pkl')
joblib.dump(df, '../models/medicine_dataframe.pkl')

print("All models and dataset successfully saved in '../models/' folder!")

All models and dataset successfully saved in '../models/' folder!
